# Assignment 02 – Reproduction of the baseline: YOLO smoke & fire detection
**Paper:** Gonçalves, Ghali, Akhloufi (2024), *YOLO-Based Models for Smoke and Wildfire Detection in Ground and Aerial Images*, Fire 7(4):140 – doi:10.3390/fire7040140  
**Student:** Sharjeel Tahir (01-134232-235) · BSCS-7A · Deep Learning  

Run top to bottom on a GPU runtime (Colab: *Runtime → Change runtime type → GPU*; Kaggle: *Accelerator → GPU*).  
Set `REPO_URL` to your own GitHub repository (or upload this folder to the notebook environment).

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/<your-username>/wildfire-yolo-repro"   # <- change
import os
if not os.path.exists("src"):
    !git clone $REPO_URL repo && %cd repo
!pip install -q -r requirements.txt
!nvidia-smi

## 1. Dataset download
**D-Fire** (images + YOLO labels): <https://github.com/gaiasd/DFireDataset> (Google Drive link in its README).  
**WSDY**: <https://www.kaggle.com/datasets/ahemateja19bec1025/wildfiresmokedatasetyolo>  

Datasets are *not* stored in the GitHub repo. Put them under `data/`:
`data/D-Fire/{train,test}/{images,labels|det_labels}` and `data/WSDY/...`.
On Kaggle, add the datasets as *Input* and point `DFIRE_ROOT` / `WSDY_ROOT` to `/kaggle/input/...`.

In [ ]:
# Option A (Colab): download the D-Fire Google-Drive folder (link taken from the dataset's README). May be rate-limited.
# import gdown; gdown.download_folder("https://drive.google.com/drive/folders/1DWgsQLVgkkLM8m-VcugHNpD5WYDbjYp5", output="data/D-Fire_download", quiet=False)
# then unzip so that data/D-Fire/train/images etc. exist.

# Option B: upload / mount manually and set the paths:
DFIRE_ROOT = "data/D-Fire"
WSDY_ROOT  = "data/WSDY"       # Kaggle: unzip the dataset here (or use the /kaggle/input path)

## 2. Task 1 – dataset preparation, statistics, visualisations

In [ ]:
!python src/prepare_dataset.py dfire --root $DFIRE_ROOT --out data/prepared
!python src/prepare_dataset.py wsdy  --root $WSDY_ROOT  --out data/prepared

In [ ]:
from IPython.display import Image, display
for f in ["dfire_samples","dfire_boxes_per_class","dfire_image_categories","dfire_object_size_hist","wsdy_samples"]:
    display(Image(f"figures/{f}.png"))
# CHECK the sample grid: boxes on flames must be labelled 'fire'. If class names look swapped,
# re-run prepare_dataset.py with --names "0:smoke,1:fire" (or the reverse).

## 3. Task 2 & 3 – baseline training (paper settings: 640 px, 100 epochs, batch 16, SGD lr 0.01, momentum 0.937, wd 5e-4, COCO-pretrained)
Hyper-parameters live in `configs/baseline.yaml` (each marked `[PAPER]` or `[ASSUME]`).  
Training D-Fire YOLOv8s for 100 epochs takes several hours on a free GPU – use `--resume` after a disconnect.  
If your compute budget is smaller, run fewer epochs (e.g. `--epochs 30`) and **report it as a deviation**.

In [ ]:
!python src/train.py --data data/prepared/dfire/dfire.yaml --model yolov8n.pt --name dfire_yolov8n

In [ ]:
!python src/train.py --data data/prepared/dfire/dfire.yaml --model yolov8s.pt --name dfire_yolov8s

In [ ]:
!python src/train.py --data data/prepared/wsdy/wsdy.yaml --model yolov8n.pt --name wsdy_yolov8n
!python src/train.py --data data/prepared/wsdy/wsdy.yaml --model yolov8s.pt --name wsdy_yolov8s

In [ ]:
# If the session died:  !python src/train.py --data ... --name dfire_yolov8s --resume

## 4. Task 4 – evaluation on the held-out test split

In [ ]:
for ds in ["dfire","wsdy"]:
    for m in ["yolov8n","yolov8s"]:
        !python src/evaluate.py --data data/prepared/{ds}/{ds}.yaml --weights runs/{ds}_{m}/weights/best.pt --name {ds}_{m}

In [ ]:
for ds in ["dfire","wsdy"]:
    for m in ["yolov8n","yolov8s"]:
        !python src/make_figures.py curves --run runs/{ds}_{m} --name {ds}_{m}
!python src/make_figures.py compare

In [ ]:
print(open("results/comparison_table.md").read())
display(Image("figures/map50_original_vs_reproduced.png"))

## 5. Save everything
Download/commit `results/`, `figures/`, `configs/`, `runs/*/results.csv`, `runs/*/experiment_config.json`.  
Large `.pt` weights should go to Google Drive / Kaggle Output / a GitHub Release, not into git.

In [ ]:
!zip -qr results_bundle.zip results figures configs runs/*/results.csv runs/*/experiment_config.json runs/*/args.yaml 2>/dev/null; ls -lh results_bundle.zip